In [ ]:
import polars as pl
import numpy as np

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
df = pl.read_csv("../data/stockdata3.csv")

In [ ]:
stocks = [c for c in df.columns if c != "day" and c != "timestr"]
returns = [f"r{s}" for s in stocks]

In [ ]:
INTRA = 0
EXTRA = 1


def get_ret(df, s):
    return (
        df.sort("datetime")
        .fill_nan(None)
        .select(
            [
                pl.col("day"),
                pl.col("datetime"),
                ((pl.col(s) / pl.col(s).shift(1)).log(base=10) * 1e4).alias(f"r{s}"),
            ]
        )
    )


def get_ret_all(df, s):
    ret = get_ret(df, s[0])
    for stock in s[1:]:
        print(stock)
        ret = ret.join(get_ret(df, stock), on="datetime", how="inner")

    return ret


def add_ret(df):
    return (
        df.sort("datetime")
        .fill_nan(None)
        .with_columns(
            [
                ((pl.col(s) / pl.col(s).shift(1)).log(base=10) * 1e4).alias(f"r{s}")
                for s in stocks
            ]
        )
        .with_columns(
            [
                pl.when(pl.col("day").diff() == 0)
                .then(pl.lit(INTRA))
                .otherwise(pl.lit(EXTRA))
                .alias("return_type")
            ]
        )
    )

In [ ]:
def daily_vol_from_ret(df, returns):
    if isinstance(returns, str):
        return df.group_by("day").agg(pl.col(returns).pow(2).sum().sqrt())
    else:
        return df.group_by("day").agg(
            [
                (pl.col(s1) * pl.col(s2)).sum().sqrt().alias(f"")
                for s1 in returns
                for s2 in returns
            ]
        )


def daily_vol(df, stocks):
    if isinstance(stocks, str):
        stocks = [stocks]
    returns = [f"r{s}" for s in stocks]
    vols = daily_vol_from_ret(get_ret(df, stocks[0]), returns[0])
    for s, r in zip(stocks[1:], returns[1:]):
        vols = vols.join(daily_vol_from_ret(get_ret(df, s), r), on="day")
    return vols

In [ ]:
# deal with c: likely split
def normalize_c(df):
    idxs = df.filter(pl.col("c").diff().abs() > 0.4 * pl.col("c"))["index"]
    if len(idxs) > 0:
        index = idxs[0]
    else:
        return df
    print(index)
    return df.with_columns(
        pl.when(pl.col("index") < index)
        .then(pl.col("c") / 2)
        .otherwise(pl.col("c"))
        .alias("c")
    )

In [ ]:
# deal with c: likely split
def mask_micro_noise(col, thd=100):
    ratio = thd / 1e4  # bps
    ret = (pl.col(col) / pl.col(col).shift(1)).log(base=10)
    mask = (
        (ret * ret.shift(-1) < 0) & (ret.abs() > ratio) & (ret.shift(-1).abs() > ratio)
    )
    return mask


def mask_denoised(col, thd=100):
    return ~mask_micro_noise(col, thd)


# def clean_d(df):
#     return df

In [ ]:
def downsample(df, freq):
    return df.group_by_dynamic("datetime", every=freq, group_by="day").agg(
        pl.all().last()
    )

## load data

In [ ]:
df = (
    df.with_columns(
        (pl.date(2000, 1, 1) + pl.duration(days=pl.col("day") - 1))
        .dt.combine(pl.col("timestr").str.to_time("%H:%M:%S"))
        .alias("datetime")
    )
    if "datatime" not in df.columns
    else df
)
display(df.select("day", "timestr", "datetime"))
df = df.with_row_index() if "index" not in df.columns else df

In [ ]:
# clean a and d
df = df.with_columns(
    pl.when(pl.col("a") == 0).then(None).otherwise(pl.col("a")).alias("a"),
    pl.when(pl.col("d") == 1).then(None).otherwise(pl.col("d")).alias("d"),
)
# clean c
df = normalize_c(df)
# filter d
NOISE_THD = 100  # 100 bps
df = df.filter(mask_denoised("d", NOISE_THD))

In [ ]:
# categories:
stocks_normal = ["a", "c", "e"]  # leave it as is
stocks_downsample = ["b", "d"]  # down sample to 10min
stocks_jump = ["f"]  # special treat, do Poisson jump or monthly close-close volatility

In [ ]:
display(downsample(df, "10m"))

In [ ]:
vols = daily_vol(df, stocks_normal + stocks_jump)
vols = vols.join(daily_vol(downsample(df, "10m"), stocks_downsample), on="day")

In [ ]:
vols = vols.sort("day")

In [ ]:
vols